# Module 4: Designing a Vector Search System

Vector search systems rarely break at the distance metric. They break because nobody decided what the payload had to hold until a few million documents were already ingested. This module is about the decisions that are cheap to make now and expensive to revisit later.

## What you will do

1. See the five layers of the stack as a mental checklist.
2. Design and build a multilingual news search system by answering five questions.
3. Practice filtering the way production systems do it, including the mistake that fails silently.
4. Assemble a production RAG retrieval pipeline.

**Tip for Colab:** run the cells top to bottom. The first install cell takes a minute because it downloads the embedding models.

Companion notebook to the [Module 4 lesson](https://qdrant.tech/course/beginners/module-4/).

## Setup

We install `qdrant-client` with the FastEmbed extra. FastEmbed gives us local, CPU friendly embedding models so the notebook produces real vectors rather than random numbers.

In [ ]:
!pip install -q "qdrant-client[fastembed]" 

## 1. The Layers of the Stack

Every vector search system, from a notebook prototype to a deployment serving millions of queries, is built from the same five layers. When something is slow, wrong, or expensive, the first diagnostic question is always the same: which layer is the problem in?

- **Query layer**: turns user intent into a search. Embedding the query, choosing dense vs sparse vs hybrid, fusing results, setting limits.
- **Indexing layer**: makes that search fast, through the HNSW graph for vectors and payload indexes for the fields you filter on. Get this wrong and results are usually still correct, just slow.
- **Storage layer**: holds the points themselves, the vectors, payloads, and IDs, across memory and disk.
- **Knowledge layer**: shapes the data before it ever becomes a vector: chunking, embedding model choice, payload schema. No amount of tuning elsewhere fixes a mistake here. Garbage in, garbage retrieved.
- **Distribution layer**: spreads the system across more than one machine: sharding, replication, multi node clusters. You will not need it on day one.

### Key insight

Every design decision belongs to a layer. "Add a payload index" is an indexing decision. "Switch to a multilingual embedding model" is a knowledge decision. "Move to three nodes" is a distribution decision. Once you sort decisions into layers, intimidating architecture diagrams become checklists.

## 2. Worked Example: Designing a Multilingual News Search System

Here is the brief, the kind you would get on a real project:

> Analysts at a research firm need to search global news that arrives continuously, in many languages. They ask in English ("port congestion in Southeast Asia") but expect matches from any language, and they scope every search by country, topic, date range, and source. Some queries name one specific thing, a company ticker or a ship name, that has to match exactly.

Five questions turn it into a design, and each answer is a decision that belongs to one layer.

### Question 1: What do the queries look like?

Both natural language and exact tokens. "Port congestion in Southeast Asia" is semantic intent, which is dense territory. "MAERSK-B.CO delisting" is an exact token that dense search will blur into neighboring tickers. That is the SKU problem from Module 3.

**Decision:** hybrid from the start. Two named vectors on every point, one dense and one sparse, with their rankings fused at query time. This is a query layer decision.

In [ ]:
import warnings
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

# Local mode: an in-process Qdrant, ideal for prototyping, notebooks and teaching.
# It is a Python reimplementation rather than the engine: search is exact instead
# of approximate, and payload indexes have no effect. Everything we write below is
# still the right shape for a real server.
client = QdrantClient(":memory:")

# Knowledge layer decision made concrete: a MULTILINGUAL dense model, so an
# English query can retrieve a Japanese or Vietnamese article with no translation.
# We use a small multilingual model to keep the Colab download fast. Production
# would reach for something like multilingual-e5-large.
dense_model = TextEmbedding("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# A sparse model gives us exact-token matching alongside dense semantics.
# BM25 is lightweight and language agnostic, a good fit for mixed-language news.
sparse_model = SparseTextEmbedding("Qdrant/bm25")

# Ask the model for its dimensionality rather than hardcoding it, so the
# collection config can never drift out of sync with the model.
DENSE_DIM = len(list(dense_model.embed(["hello"]))[0])
print("Dense vector dimension:", DENSE_DIM)

Dense vector dimension: 384


Now create the collection with **named vectors**: one dense, one sparse. Naming them lets a single point carry both, and lets a single query fuse both.

The sparse config needs `modifier=models.Modifier.IDF`. BM25-style sparse vectors store only term frequency, and Qdrant applies the inverse-document-frequency half of the formula at query time. Leave the modifier off and you are not scoring BM25, you are scoring raw term counts.

In [ ]:
client.create_collection(
    collection_name="news",
    vectors_config={
        "dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF   # required for BM25 scoring
        ),
    },
)
print("Collection 'news' created.")

Collection 'news' created.


### Question 2: What must the system filter on?

From the brief: country, topic, date range, and source. These are hard rules, not similarity signals. An analyst scoping to "Vietnam, last seven days" means exactly that. Hard rules go in the payload, and every field we filter on gets a payload index.

The schema, decided now, before ingestion:

```yaml
payload:
  country: string         # indexed
  language: string        # indexed
  topic: string           # indexed
  source: string          # indexed
  tenant_id: string       # indexed, and marked as a tenant field
  published_at: datetime  # indexed
  summary: string         # not indexed: returned, never filtered
```

Create the indexes now too, in the same setup step, before a single point is uploaded. This ordering is not a style preference. Qdrant extends the HNSW graph with extra edges derived from indexed payload values, and it can only add those edges for indexes that already exist when the graph is built. Create a payload index after ingesting and you have to rebuild the HNSW index to get any benefit from it.

Note `tenant_id` is in this list even though multitenancy does not come up until Section 3. That is the point: it is nearly free now and expensive later.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # local mode warns that indexes do nothing here

    for field in ["country", "language", "topic", "source"]:
        client.create_payload_index(
            collection_name="news",
            field_name=field,
            field_schema=models.PayloadSchemaType.KEYWORD,
        )

    client.create_payload_index(
        collection_name="news",
        field_name="published_at",
        field_schema=models.PayloadSchemaType.DATETIME,
    )

    # is_tenant tells Qdrant this field identifies tenants, so it can keep each
    # tenant's data close together on disk. Supported for keyword and uuid.
    client.create_payload_index(
        collection_name="news",
        field_name="tenant_id",
        field_schema=models.KeywordIndexParams(
            type=models.KeywordIndexType.KEYWORD,
            is_tenant=True,
        ),
    )

print("Payload indexes created before ingestion: country, language, topic, source, published_at, tenant_id.")

Payload indexes created before ingestion: country, language, topic, source, published_at, tenant_id.


Skipping an index costs more than a slow scan. The payload index is also what lets Qdrant estimate how many points a filter will match, and that estimate is what the query planner uses to choose a strategy at all. Without it the planner is guessing, and it can fall back to comparing the query against every vector in the collection.

Because the query still returns results, just slowly, a missing index can sit in production unnoticed for months. Strict mode closes that gap: set `unindexed_filtering_retrieve` to `false` and Qdrant rejects any query that filters on an unindexed field instead of quietly degrading. Qdrant Cloud applies this by default.

### Question 3: What is the workload shape?

Millions of articles, text only for now, arriving continuously. Analysts expect this morning's news to be searchable this morning.

**Decisions:** one collection, and continuous upserts rather than periodic rebuilds. Two things still need designing. The initial backfill of millions of articles is a bulk load, not a stream, so batch the upserts and consider disabling indexing for the duration so the optimizer builds the graph once at the end. And Qdrant does index as it ingests, but not instantly: the optimizer builds an HNSW index for a segment only once that segment passes the indexing threshold, and unindexed segments are served by full scan in the meantime.

Here is a small, multilingual sample that stands in for that stream. Note the mix of languages and the exact tokens (a ticker, a ship name) hiding in the text.

In [ ]:
articles = [
    {
        "country": "VN", "language": "vi", "topic": "shipping", "source": "reuters",
        "published_at": "2026-07-20T08:00:00Z", "tenant_id": "asia-desk",
        "summary": "Tac nghen cang tai Hai Phong khi luong hang tang manh.",
        "text": "Cang Hai Phong o Viet Nam bi tac nghen nghiem trong trong tuan nay khi luong container tang vot, gay cham tre cho tau thuyen.",
    },
    {
        "country": "JP", "language": "ja", "topic": "shipping", "source": "nikkei",
        "published_at": "2026-07-21T09:30:00Z", "tenant_id": "asia-desk",
        "summary": "東南アジアの港湾混雑が輸送を遅らせている。",
        "text": "東南アジアの主要港で混雑が深刻化し、コンテナ船の到着が遅れている。特にベトナムとタイの港で影響が大きい。",
    },
    {
        "country": "CN", "language": "zh", "topic": "logistics", "source": "caixin",
        "published_at": "2026-07-19T11:00:00Z", "tenant_id": "asia-desk",
        "summary": "上海港物流吞吐量创新高。",
        "text": "上海港本月物流吞吐量创下新高，港口运营商正在扩大堆场以缓解拥堵压力。",
    },
    {
        "country": "DK", "language": "en", "topic": "markets", "source": "reuters",
        "published_at": "2026-07-22T07:15:00Z", "tenant_id": "europe-desk",
        "summary": "Maersk shares move on delisting speculation.",
        "text": "Shares tied to the ticker MAERSK-B.CO moved sharply amid speculation about a possible delisting of a subsidiary vehicle.",
    },
    {
        "country": "SG", "language": "en", "topic": "shipping", "source": "straits-times",
        "published_at": "2026-07-22T10:45:00Z", "tenant_id": "asia-desk",
        "summary": "Vessel Ever Given reroutes through Singapore.",
        "text": "The container ship Ever Given was rerouted through the Port of Singapore this week to avoid congestion further north.",
    },
    {
        "country": "US", "language": "en", "topic": "markets", "source": "press-release-wire",
        "published_at": "2026-06-30T12:00:00Z", "tenant_id": "americas-desk",
        "summary": "Company announces quarterly logistics earnings.",
        "text": "A logistics operator announced quarterly earnings, citing steady demand across North American freight corridors.",
    },
    {
        "country": "TH", "language": "th", "topic": "shipping", "source": "bangkok-post",
        "published_at": "2026-07-18T06:20:00Z", "tenant_id": "asia-desk",
        "summary": "ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น",
        "text": "ท่าเรือแหลมฉบังของไทยเผชิญความแออัดมากขึ้นเนื่องจากปริมาณตู้คอนเทนเนอร์ที่เพิ่มสูงขึ้นในสัปดาห์นี้",
    },
    {
        "country": "DE", "language": "de", "topic": "logistics", "source": "handelsblatt",
        "published_at": "2026-07-15T14:10:00Z", "tenant_id": "europe-desk",
        "summary": "Hamburger Hafen meldet Verzoegerungen.",
        "text": "Der Hamburger Hafen meldet Verzoegerungen bei der Abfertigung, da die Zahl der ankommenden Schiffe deutlich gestiegen ist.",
    },
]
print(f"{len(articles)} sample articles across {len(set(a['language'] for a in articles))} languages.")

8 sample articles across 6 languages.


Embed each article into both a dense and a sparse vector, and upsert them as points. In production this same loop runs continuously as articles arrive.

In [ ]:
def to_sparse_vector(text, is_query=False):
    """Convert text to a Qdrant SparseVector using BM25.
    BM25 scores queries and documents slightly differently, so we use
    query_embed for queries and embed for documents."""
    emb = next(sparse_model.query_embed(text)) if is_query else next(sparse_model.embed([text]))
    return models.SparseVector(indices=emb.indices.tolist(), values=emb.values.tolist())

points = []
for i, art in enumerate(articles):
    dense_vec = next(dense_model.embed([art["text"]])).tolist()
    sparse_vec = to_sparse_vector(art["text"])
    points.append(
        models.PointStruct(
            id=i,
            vector={"dense": dense_vec, "sparse": sparse_vec},
            payload=art,
        )
    )

client.upsert(collection_name="news", points=points)
print(f"Upserted {len(points)} points. Collection count:", client.count("news").count)

Upserted 8 points. Collection count: 8


### Question 4: What does the retrieval pipeline look like?

Start with the simplest pipeline that fits the query analysis: hybrid (from Question 1) plus filters (from Question 2), fused by rank. No reranker yet. Add complexity when data proves it is needed, never in advance.

The `query_points` call below does it all in one round trip:

- Two `Prefetch` branches, one per named vector, each pulling 50 candidates.
- `RrfQuery` to merge the two candidate lists into one ranking.
- The filter passed **into each prefetch**, so both retrievers only ever consider valid points.

That last point is the one to slow down on, and Section 3 breaks it on purpose to show why.

Notice the payoff of the multilingual model: we query in **English** and retrieve articles written in Japanese, Vietnamese, Thai, and more, with no translation step.

In [ ]:
def search(query_text, query_filter=None, limit=5):
    """Hybrid search: dense + sparse prefetch, fused by rank, filter in each branch."""
    dense_q = next(dense_model.query_embed(query_text)).tolist()
    sparse_q = to_sparse_vector(query_text, is_query=True)
    response = client.query_points(
        collection_name="news",
        prefetch=[
            models.Prefetch(query=dense_q,  using="dense",
                            filter=query_filter, limit=50),
            models.Prefetch(query=sparse_q, using="sparse",
                            filter=query_filter, limit=50),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=limit,
    )
    return response.points

def show(results):
    if not results:
        print("   (no results)")
    for r in results:
        p = r.payload
        print(f"[{r.score:.4f}] {p['country']}/{p['language']:<2} {p['topic']:<9} | {p['summary']}")

print("Query (English): 'port congestion in Southeast Asia'\n")
show(search("port congestion in Southeast Asia"))

Query (English): 'port congestion in Southeast Asia'

[0.7500] SG/en shipping  | Vessel Ever Given reroutes through Singapore.
[0.5000] JP/ja shipping  | 東南アジアの港湾混雑が輸送を遅らせている。
[0.3333] TH/th shipping  | ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น
[0.2000] CN/zh logistics | 上海港物流吞吐量创新高。
[0.1667] DE/de logistics | Hamburger Hafen meldet Verzoegerungen.


Now the exact-token case. A dense-only search would blur `MAERSK-B.CO` into semantically similar finance text. The sparse half of the hybrid locks onto the literal token, and rank fusion pushes the correct article to the top. This is why we chose hybrid from the start.

In [ ]:
print("Query (exact token): 'MAERSK-B.CO delisting'\n")
show(search("MAERSK-B.CO delisting"))

Query (exact token): 'MAERSK-B.CO delisting'

[1.0000] DK/en markets   | Maersk shares move on delisting speculation.
[0.3333] DE/de logistics | Hamburger Hafen meldet Verzoegerungen.
[0.2500] VN/vi shipping  | Tac nghen cang tai Hai Phong khi luong hang tang manh.
[0.2000] SG/en shipping  | Vessel Ever Given reroutes through Singapore.
[0.1667] TH/th shipping  | ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น


Reciprocal Rank Fusion is the right default here because it works from ranks rather than raw scores, so it does not care that cosine similarity and BM25 live on different scales. Two alternatives are worth knowing once you can measure quality: weighted RRF, which favours the stronger retriever via `weights` on `models.Rrf()`, and Distribution-Based Score Fusion, `models.FusionQuery(fusion=models.Fusion.DBSF)`, which normalizes each retriever's score distribution instead. Neither reliably beats the other, so choose with an evaluation set rather than by reputation.

### Question 5: What are the deployment constraints?

A research firm with a small engineering team, no data residency restrictions, and a "please do not page us at night" budget points to a **managed** deployment. The key idea: the design and the deployment mode are independent decisions. Everything above runs unchanged whether the backend is this in-process client, a Docker container, or Qdrant Cloud.

### The design on one page

| Question | Answer for This System | Layer |
|----------|------------------------|-------|
| Query type | Mixed semantic + exact, so hybrid with Reciprocal Rank Fusion | Query |
| Filter scope | country, topic, source, date, indexed before ingestion | Knowledge, indexing |
| Workload shape | Millions of text chunks: bulk backfill, then continuous ingestion | Storage, knowledge |
| Pipeline | Hybrid + per-prefetch filters, multilingual model, no reranker yet | Query, knowledge |
| Deployment | Managed, design independent of the choice | Distribution |

### Key insight

Every piece of this design already appeared in an earlier module: the Module 2 pipeline, the Module 3 hybrid pattern, and a payload schema. What is new is the order of operations. You decided what to filter on and created the indexes before the first point went in. That ordering is the difference between a system that scales and one that gets re-ingested three times.

## 3. Filtering

Filtering is the feature that decides whether your results are **correct**, not an accessory to similarity.

### How Qdrant combines filters with vector search

The naive approach is post-filtering: retrieve the top K by similarity, then discard whatever fails the filter. With a selective filter, say one country out of 200, the top K can contain zero valid results, and there is no K that guarantees correctness.

Qdrant does not work that way. A query planner chooses a strategy for each segment, based on the estimated cardinality of the filter and which payload indexes exist:

- When the filter matches a large share of the collection, Qdrant walks the HNSW graph as usual and skips points that fail the filter during traversal.
- When the filter matches very few points, it can skip the graph entirely and retrieve through the payload index, which is cheaper at that selectivity.
- When a segment is small enough, a full scan wins outright.

The middle ground is the hard case, because a strict filter can disconnect the HNSW graph and leave relevant points unreachable. Qdrant handles it by extending the graph with additional edges derived from indexed payload values, which is why the indexes had to exist before ingestion. Those extra edges are added per index, not per combination of indexes, so two or more highly selective filters can still reach a disconnected component. The ACORN algorithm (v1.16) exists for that case.

So: filtering on every query is safe and fast, as long as every filtered field is indexed and the indexes existed before ingestion. It is not free, and combinations of highly selective filters deserve measurement rather than assumption.

### The filter toolbox

| Condition | Logic | Example |
|-----------|-------|---------|
| must | AND, all conditions true | country = VN AND topic = shipping |
| should | OR, at least one true (`min_should` sets a higher minimum) | topic = shipping OR topic = logistics |
| must_not | Exclude matches | Exclude source = press-release-wire |
| Range | Numeric or datetime bounds | published_at within the last seven days |
| MatchAny | Value in a set | language in [ja, zh, th, vi] |
| Geo | Radius, bounding box, or polygon | Events within 100 km of a port |

The cells below run several of these against the news collection.

In [ ]:
# must: AND. Only Vietnamese shipping news.
f_must = models.Filter(
    must=[
        models.FieldCondition(key="country", match=models.MatchValue(value="VN")),
        models.FieldCondition(key="topic", match=models.MatchValue(value="shipping")),
    ]
)
print("must (country=VN AND topic=shipping):")
show(search("port congestion", query_filter=f_must))

must (country=VN AND topic=shipping):


[0.5000] VN/vi shipping  | Tac nghen cang tai Hai Phong khi luong hang tang manh.


In [ ]:
# should: OR. Either shipping or logistics topics.
f_should = models.Filter(
    should=[
        models.FieldCondition(key="topic", match=models.MatchValue(value="shipping")),
        models.FieldCondition(key="topic", match=models.MatchValue(value="logistics")),
    ]
)
print("should (topic=shipping OR topic=logistics):")
show(search("congestion at ports", query_filter=f_should))

should (topic=shipping OR topic=logistics):


[0.7000] SG/en shipping  | Vessel Ever Given reroutes through Singapore.
[0.5000] JP/ja shipping  | 東南アジアの港湾混雑が輸送を遅らせている。
[0.3333] CN/zh logistics | 上海港物流吞吐量创新高。
[0.2500] TH/th shipping  | ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น
[0.1667] DE/de logistics | Hamburger Hafen meldet Verzoegerungen.


In [ ]:
# must_not + Range + MatchAny combined, evaluated together while the search runs.
f_combined = models.Filter(
    must=[
        models.FieldCondition(
            key="published_at",
            range=models.DatetimeRange(gte="2026-07-15T00:00:00Z"),
        ),
        models.FieldCondition(
            key="language",
            match=models.MatchAny(any=["ja", "zh", "th", "vi"]),
        ),
    ],
    must_not=[
        models.FieldCondition(key="source", match=models.MatchValue(value="press-release-wire")),
    ],
)
print("recent AND asian-language AND not a press-release wire:")
show(search("port congestion in Southeast Asia", query_filter=f_combined))

recent AND asian-language AND not a press-release wire:
[0.5000] JP/ja shipping  | 東南アジアの港湾混雑が輸送を遅らせている。
[0.3333] TH/th shipping  | ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น
[0.2500] CN/zh logistics | 上海港物流吞吐量创新高。
[0.2000] VN/vi shipping  | Tac nghen cang tai Hai Phong khi luong hang tang manh.


### Common mistake: filters in the wrong place

Every filtered search above worked because `search()` passes the filter into each `Prefetch`. Move it to a top-level `query_filter` and it silently stops constraining the retrievers.

Whenever a query has at least one prefetch, Qdrant runs the prefetches first and applies the main query to their results. A top-level filter therefore never reaches them: each searches the whole collection, returns its 50 candidates, and the filter only trims the fused set at the end. That is post-filtering, with exactly the failure mode described above.

Below, the same selective filter runs both ways. The correct version returns only Vietnamese shipping news. The broken version does not, and nothing raises an error.

In [ ]:
def search_wrong(query_text, query_filter=None, limit=5):
    """Identical to search(), except the filter sits at the top level."""
    dense_q = next(dense_model.query_embed(query_text)).tolist()
    sparse_q = to_sparse_vector(query_text, is_query=True)
    return client.query_points(
        collection_name="news",
        prefetch=[
            models.Prefetch(query=dense_q,  using="dense",  limit=50),
            models.Prefetch(query=sparse_q, using="sparse", limit=50),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        query_filter=query_filter,    # too late: the prefetches already ran
        limit=limit,
    ).points

print("CORRECT  filter inside each prefetch (country=VN AND topic=shipping):")
show(search("port congestion", query_filter=f_must))
print()
print("BROKEN   same filter at the top level:")
show(search_wrong("port congestion", query_filter=f_must))

CORRECT  filter inside each prefetch (country=VN AND topic=shipping):
[0.5000] VN/vi shipping  | Tac nghen cang tai Hai Phong khi luong hang tang manh.

BROKEN   same filter at the top level:
[0.7000] SG/en shipping  | Vessel Ever Given reroutes through Singapore.
[0.5000] JP/ja shipping  | 東南アジアの港湾混雑が輸送を遅らせている。
[0.3333] CN/zh logistics | 上海港物流吞吐量创新高。
[0.2500] TH/th shipping  | ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น
[0.1667] DE/de logistics | Hamburger Hafen meldet Verzoegerungen.


The rule is short. No prefetch, use `query_filter`. Prefetch, put the filter in every prefetch.

### A special case: scoping by user or tenant

In almost any multi user product you must scope every query to one user's or one customer's data. The instinct is a collection per user, which becomes millions of collections and is operationally unmanageable. The standard pattern instead:

1. Add a `tenant_id` payload field to every point at ingestion. (Done, above.)
2. Create a payload index on it with `is_tenant=True`. (Done, before ingestion.)
3. Filter on it at every query. Never omit it.

Steps 1 and 3 alone are already correct: they just get slower than they need to as tenant count grows. Step 2 is what keeps it fast, and it is the one people skip.

In [ ]:
def tenant_search(query_text, tenant_id, limit=5):
    return search(
        query_text,
        query_filter=models.Filter(
            must=[models.FieldCondition(key="tenant_id", match=models.MatchValue(value=tenant_id))]
        ),
        limit=limit,
    )

print("europe-desk view of 'port delays':")
show(tenant_search("port delays", tenant_id="europe-desk"))
print("\nasia-desk view of 'port delays':")
show(tenant_search("port delays", tenant_id="asia-desk"))

europe-desk view of 'port delays':
[0.5000] DE/de logistics | Hamburger Hafen meldet Verzoegerungen.
[0.3333] DK/en markets   | Maersk shares move on delisting speculation.

asia-desk view of 'port delays':
[0.7000] SG/en shipping  | Vessel Ever Given reroutes through Singapore.
[0.5000] JP/ja shipping  | 東南アジアの港湾混雑が輸送を遅らせている。
[0.3333] TH/th shipping  | ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น
[0.2500] CN/zh logistics | 上海港物流吞吐量创新高。
[0.1667] VN/vi shipping  | Tac nghen cang tai Hai Phong khi luong hang tang manh.


### Key insight

Design the payload schema before you ingest, driven by one question: what will I need to filter on? Time, geography, identity, permissions, and status flags are the usual suspects.

Adding a payload *field* later is easy. Adding a payload *index* later means rebuilding the HNSW graph, and discovering at query time that you never stored `language` at all means re-ingesting everything.

## 4. The Production RAG Pipeline

Retrieval-Augmented Generation (RAG) retrieves relevant passages from a vector search engine and hands them to an LLM as context, so the model answers from your data instead of relying only on what it memorized during training.

The production shape, using everything above:

1. **Query understanding:** extract hard constraints (dates, country, topic) into a filter. Embed the query as dense and sparse vectors.
2. **Hybrid retrieval:** dense + sparse prefetch, each carrying the filter, fused by rank. One `query_points` call.
3. **Optional reranking:** a cross-encoder scores the top candidates and keeps the best few. Add this only when evaluation shows fused results need refinement.
4. **LLM generation:** the top passages go in as context, the model generates the answer with sources.

### Rule of thumb

When RAG quality disappoints, improve step 2 before reaching for a bigger model in step 4. Retrieval quality caps answer quality: the model cannot cite a passage it never received.

The cell below implements steps 1, 2, and the prompt assembly for step 4. It runs with no API key and prints the exact prompt an LLM would receive.

In [ ]:
def rag_context(question, query_filter=None, k=3):
    """Steps 1 and 2: retrieve the top-k passages for a question."""
    hits = search(question, query_filter=query_filter, limit=k)
    passages = []
    for h in hits:
        p = h.payload
        passages.append(f"- ({p['source']}, {p['country']}, {p['published_at'][:10]}) {p['summary']}")
    return "\n".join(passages), hits

def build_prompt(question, query_filter=None, k=3):
    context, hits = rag_context(question, query_filter=query_filter, k=k)
    prompt = (
        "Answer the question using only the sources below. Cite the source name.\n\n"
        f"Sources:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    return prompt, hits

prompt, hits = build_prompt("What is happening with port congestion in Southeast Asia?")
print(prompt)

Answer the question using only the sources below. Cite the source name.

Sources:
- (straits-times, SG, 2026-07-22) Vessel Ever Given reroutes through Singapore.
- (nikkei, JP, 2026-07-21) 東南アジアの港湾混雑が輸送を遅らせている。
- (bangkok-post, TH, 2026-07-18) ความแออัดที่ท่าเรือแหลมฉบังเพิ่มขึ้น

Question: What is happening with port congestion in Southeast Asia?
Answer:


In [ ]:
# OPTIONAL step 4: hand the prompt to a real LLM. This block is inert unless you
# add an API key, so the notebook still runs end to end without one.
#
# import os
# from anthropic import Anthropic
# llm = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
# msg = llm.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=300,
#     messages=[{"role": "user", "content": prompt}],
# )
# print(msg.content[0].text)

print("Prompt is ready. Uncomment the block above and set an API key to generate an answer.")

Prompt is ready. Uncomment the block above and set an API key to generate an answer.


## What's next: Module 5

The capstone extends this system to three modalities at once: news, audio, and satellite imagery on shared points, clustered into risk themes and queried across languages.

[Continue to Module 5](https://qdrant.tech/course/beginners/module-5/)